# PCA & Manifold Learning — Real Datasets
## Notebook 2: Real-World Applications

**Datasets Used:**
- **Iris** (4 features) — classic PCA visualization
- **Digits** (64 features) — image compression & reconstruction
- **Breast Cancer** (30 features) — high-dim reduction for ML
- **Capstone:** Wine Quality dataset (minimal plotting)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.datasets import load_iris, load_digits, load_breast_cancer, load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded!")

---
## Part 1: PCA on Iris Dataset (4D → 2D)

**Dataset:** 150 flowers, 4 features (sepal length, sepal width, petal length, petal width), 3 classes.

**Goal:** Reduce 4D → 2D for visualization while preserving class separation.

In [ ]:
# Load Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Shape: {X_iris.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")
print(f"\nClass counts: {np.bincount(y_iris)}")

In [ ]:
# PCA on Iris: 4D -> 2D
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

pca_iris = PCA(n_components=2)
X_iris_pca = pca_iris.fit_transform(X_iris_scaled)

print(f"Original: {X_iris_scaled.shape} -> PCA: {X_iris_pca.shape}")
print(f"Explained variance ratio: {pca_iris.explained_variance_ratio_.round(4)}")
print(f"Total variance retained: {pca_iris.explained_variance_ratio_.sum()*100:.1f}%")

# Loadings (which features contribute to each PC)
print("\nPC Loadings (how each feature contributes):")
print(f"{'Feature':<25} {'PC1':>8} {'PC2':>8}")
for name, l1, l2 in zip(feature_names, pca_iris.components_[0], pca_iris.components_[1]):
    print(f"{name:<25} {l1:>8.3f} {l2:>8.3f}")

In [ ]:
# Visualize Iris PCA
colors = ['royalblue', 'tomato', 'forestgreen']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: PCA 2D
ax = axes[0]
for i, name in enumerate(target_names):
    mask = y_iris == i
    ax.scatter(X_iris_pca[mask,0], X_iris_pca[mask,1],
               color=colors[i], label=name, alpha=0.8, s=50)
ax.set_xlabel(f"PC1 ({pca_iris.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca_iris.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title('Iris: PCA 4D → 2D', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: t-SNE
tsne_iris = TSNE(n_components=2, perplexity=20, random_state=42)
X_iris_tsne = tsne_iris.fit_transform(X_iris_scaled)
ax = axes[1]
for i, name in enumerate(target_names):
    mask = y_iris == i
    ax.scatter(X_iris_tsne[mask,0], X_iris_tsne[mask,1],
               color=colors[i], label=name, alpha=0.8, s=50)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.set_title('Iris: t-SNE 4D → 2D', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Iris Dataset: Dimensionality Reduction', fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2: PCA on Digits Dataset (64D → compressed)

**Dataset:** 1,797 handwritten digits (0-9), 8×8 pixel images = 64 features.

**Goal:** Image compression — reduce 64 dimensions, then reconstruct to see quality.

In [ ]:
# Load Digits
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Shape: {X_digits.shape}")
print(f"Each image: 8×8 = 64 pixels (features)")
print(f"Classes: {np.unique(y_digits)}")

# Show sample digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_digits[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Label: {y_digits[i]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Digits (original 8×8)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# PCA: how many components needed?
scaler_dig = StandardScaler()
X_dig_scaled = scaler_dig.fit_transform(X_digits)

pca_dig_full = PCA().fit(X_dig_scaled)
cumvar = np.cumsum(pca_dig_full.explained_variance_ratio_)

n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1

print(f"Original: 64 features")
print(f"Components for 90% variance: {n_90}  ({n_90/64*100:.0f}% of original)")
print(f"Components for 95% variance: {n_95}  ({n_95/64*100:.0f}% of original)")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 65), cumvar, 'steelblue', lw=2)
plt.axhline(0.90, color='orange', ls='--', label=f'90% → {n_90} PCs')
plt.axhline(0.95, color='red', ls='--', label=f'95% → {n_95} PCs')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Digits: Variance Explained vs Components', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Image reconstruction at different n_components
components_to_try = [2, 10, 20, 40, 64]
sample_idx = 10

fig, axes = plt.subplots(1, len(components_to_try) + 1, figsize=(14, 3))

# Original
axes[0].imshow(X_dig_scaled[sample_idx].reshape(8, 8), cmap='gray_r')
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for ax, k in zip(axes[1:], components_to_try):
    pca_k = PCA(n_components=k)
    X_comp = pca_k.fit_transform(X_dig_scaled)
    X_rec = pca_k.inverse_transform(X_comp)
    mse = np.mean((X_dig_scaled - X_rec)**2)
    var = pca_k.explained_variance_ratio_.sum() * 100
    ax.imshow(X_rec[sample_idx].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'k={k}\n{var:.0f}%', fontsize=9)
    ax.axis('off')

plt.suptitle('Digit Reconstruction at Different Compression Levels', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D visualization of digits (all 10 classes)
pca_dig_2d = PCA(n_components=2)
X_dig_2d = pca_dig_2d.fit_transform(X_dig_scaled)

tsne_dig = TSNE(n_components=2, perplexity=30, random_state=42)
X_dig_tsne = tsne_dig.fit_transform(X_dig_scaled[:500])  # 500 for speed

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.get_cmap('tab10', 10)

for digit in range(10):
    mask = y_digits == digit
    axes[0].scatter(X_dig_2d[mask,0], X_dig_2d[mask,1],
                    color=cmap(digit), alpha=0.5, s=15, label=str(digit))
axes[0].set_title(f'PCA: 64D→2D ({pca_dig_2d.explained_variance_ratio_.sum()*100:.1f}% var)', fontsize=11)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(title='Digit', ncol=2, fontsize=8, markerscale=2)
axes[0].grid(True, alpha=0.3)

for digit in range(10):
    mask = y_digits[:500] == digit
    axes[1].scatter(X_dig_tsne[mask,0], X_dig_tsne[mask,1],
                    color=cmap(digit), alpha=0.5, s=15, label=str(digit))
axes[1].set_title('t-SNE: 64D→2D (500 samples)', fontsize=11)
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
axes[1].legend(title='Digit', ncol=2, fontsize=8, markerscale=2)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Digits: PCA vs t-SNE', fontsize=13)
plt.tight_layout()
plt.show()
print("t-SNE shows much cleaner digit clusters!")

---
## Part 3: PCA for ML — Breast Cancer Dataset (30D)

**Dataset:** 569 samples, 30 features, 2 classes (malignant/benign).

**Goal:** Use PCA to reduce features, then compare ML accuracy.

In [ ]:
# Breast cancer: PCA + classification
bc = load_breast_cancer()
X_bc = bc.data
y_bc = bc.target
print(f"Shape: {X_bc.shape}, Classes: {bc.target_names}")

scaler_bc = StandardScaler()
X_bc_scaled = scaler_bc.fit_transform(X_bc)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_bc_scaled, y_bc, test_size=0.2, random_state=42, stratify=y_bc)

# Accuracy at different n_components
results = []
for k in [2, 5, 10, 15, 20, 30]:
    if k < 30:
        pca_bc = PCA(n_components=k)
        X_tr = pca_bc.fit_transform(X_train)
        X_te = pca_bc.transform(X_test)
        var = pca_bc.explained_variance_ratio_.sum() * 100
    else:
        X_tr, X_te = X_train, X_test
        var = 100.0
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_train)
    acc = accuracy_score(y_test, lr.predict(X_te)) * 100
    results.append((k, var, acc))

print(f"\n{'n_components':<15} {'Var Retained%':<18} {'Accuracy%':<12}")
print("-" * 45)
for k, var, acc in results:
    label = "(baseline)" if k == 30 else ""
    print(f"{k:<15} {var:<18.1f} {acc:<12.1f} {label}")

In [ ]:
# Plot accuracy vs n_components
ks = [r[0] for r in results]
accs = [r[2] for r in results]
vars_ret = [r[1] for r in results]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(ks, accs, 'o-', color='steelblue', lw=2, label='Accuracy %')
ax2.plot(ks, vars_ret, 's--', color='tomato', lw=2, label='Variance %')
ax1.set_xlabel('n_components')
ax1.set_ylabel('Accuracy (%)', color='steelblue')
ax2.set_ylabel('Variance Retained (%)', color='tomato')
ax1.set_title('Breast Cancer: PCA + LR Accuracy vs Compression', fontsize=12)
ax1.grid(True, alpha=0.3)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.tight_layout()
plt.show()
print("PCA keeps accuracy while reducing 30 features to ~10!")

---
## Part 4: Capstone — Wine Quality PCA Pipeline

**Dataset:** Wine quality (sklearn load_wine), 13 chemical features, 3 classes.

**Task:** Full PCA pipeline — standardize, reduce, visualize, compare models.

In [ ]:
# ============================================================
# CAPSTONE: Wine Quality PCA Pipeline
# ============================================================
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
print(f"Wine data: {X_wine.shape}, {len(wine.target_names)} classes: {wine.target_names}")

# Step 1: Scale
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# Step 2: Find n_components (95% threshold)
pca_wine_full = PCA().fit(X_wine_scaled)
cumvar_wine = np.cumsum(pca_wine_full.explained_variance_ratio_)
n_opt_wine = np.argmax(cumvar_wine >= 0.95) + 1
print(f"\nOptimal components (95% var): {n_opt_wine} from 13")
print(f"Compression: {13}D → {n_opt_wine}D ({n_opt_wine/13*100:.0f}% of features)")

# Step 3: Apply PCA
pca_wine = PCA(n_components=n_opt_wine)
X_wine_pca = pca_wine.fit_transform(X_wine_scaled)
print(f"Variance retained: {pca_wine.explained_variance_ratio_.sum()*100:.1f}%")

# Step 4: 2D for visualization
pca_wine_2d = PCA(n_components=2)
X_wine_2d = pca_wine_2d.fit_transform(X_wine_scaled)

In [ ]:
# Step 5: Classification comparison (minimal plotting in capstone)
X_tr_full, X_te_full, y_tr, y_te = train_test_split(
    X_wine_scaled, y_wine, test_size=0.25, random_state=42, stratify=y_wine)
X_tr_pca, X_te_pca = pca_wine.transform(X_tr_full), pca_wine.transform(X_te_full)

lr_full = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_full, y_tr)
lr_pca = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_pca, y_tr)

acc_full = accuracy_score(y_te, lr_full.predict(X_te_full)) * 100
acc_pca = accuracy_score(y_te, lr_pca.predict(X_te_pca)) * 100

print("=" * 45)
print("CAPSTONE RESULTS: Wine Quality")
print("=" * 45)
print(f"{'Metric':<30} {'Full':>8} {'PCA':>8}")
print("-" * 45)
print(f"{'Features':<30} {'13':>8} {n_opt_wine:>8}")
print(f"{'Accuracy (%)':<30} {acc_full:>8.1f} {acc_pca:>8.1f}")
print(f"{'Variance Retained (%)':<30} {'100.0':>8} {pca_wine.explained_variance_ratio_.sum()*100:>8.1f}")
print("=" * 45)
print(f"\nResult: PCA reduced {13} → {n_opt_wine} features")
print(f"Accuracy change: {acc_pca - acc_full:+.1f}%")
if abs(acc_pca - acc_full) < 2:
    print("Near-identical accuracy with fewer features — PCA works well here!")

# Class visualization (one minimal plot)
fig, ax = plt.subplots(figsize=(7, 5))
colors_wine = ['royalblue', 'tomato', 'forestgreen']
for i, name in enumerate(wine.target_names):
    mask = y_wine == i
    ax.scatter(X_wine_2d[mask,0], X_wine_2d[mask,1],
               color=colors_wine[i], alpha=0.8, s=40, label=name)
ax.set_xlabel(f"PC1 ({pca_wine_2d.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca_wine_2d.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title('Wine: PCA 13D → 2D Visualization', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()